# Lab 15-10 · State-Space Introduction

Four parts, each built on `statespace.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm
from statespace import ctrb, obsv, acker, observer_gain, closed_loop_response

## Part 1 · Modes in the phase plane

In [2]:
A = np.array([[0.0, 1.0], [-2.0, -3.0]])
w, V = np.linalg.eig(A)
w, V = w.real, V.real                 # real eigenvalues here; drop the zero imaginary parts
print("eigenvalues", w, " eigenvector slopes", V[1] / V[0])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for th in np.linspace(0, 2 * np.pi, 13)[:-1]:
    x0 = np.array([math.cos(th), 2 * math.sin(th)])
    X = np.array([expm(A * t) @ x0 for t in np.linspace(0, 6, 300)])
    ax.plot(X[:, 0], X[:, 1], lw=0.8)
for k in range(2):
    ax.plot([-V[0, k] * 2, V[0, k] * 2], [-V[1, k] * 2, V[1, k] * 2], "k--", lw=0.8)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-2.5, 2.5); ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.show()

eigenvalues [-1. -2.]  eigenvector slopes [-1. -2.]


## Part 2 · State feedback in ngspice

*Predict:* overshoot and peak time of $16/(s^2 + 4s + 16)$.

In [3]:
z, wn = 0.5, 4.0
print(f"predicted: overshoot {100 * math.exp(-math.pi * z / math.sqrt(1 - z * z)):.2f}%, peak at {math.pi / (wn * math.sqrt(1 - z * z)):.4f} s")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "state_feedback.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("os =", "tpk ="))])
else:
    print("ngspice not installed: skip (CI runs state_feedback.cir through tools/run_spice.py)")

predicted: overshoot 16.30%, peak at 0.9069 s
ngspice: ['os = 1.630340e+01', 'tpk = 9.069002e-01']


## Part 3 · Nearly uncontrollable

In [4]:
taus = np.linspace(1.001, 2.0, 200)
smin = []
for tau2 in taus:
    A2 = np.diag([-1.0, -1.0 / tau2]); B2 = np.array([[1.0], [1.0 / tau2]])
    smin.append(np.linalg.svd(ctrb(A2, B2), compute_uv=False)[-1])
fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(taus - 1, smin); ax.set_xlabel("tau2 - 1 ms (ms)"); ax.set_ylabel("smallest singular value")
ax.grid(True, which="both", alpha=0.3)
plt.show()
print("slope on log-log axes:", np.polyfit(np.log(taus[:50] - 1), np.log(smin[:50]), 1)[0])

slope on log-log axes: 0.9276587142944486


## Part 4 · Observer speed and noise

In [5]:
A3 = np.array([[0.0, 1.0], [0.0, -1.0]]); B3 = np.array([[0.0], [1.0]]); C3 = np.array([[1.0, 0.0]])
rng = np.random.default_rng(1)
dt, T = 1e-3, 4.0
noise = rng.normal(0, 0.01, int(T / dt))
for p in (-5, -10, -40):
    L = observer_gain(A3, C3, [p, p])
    Ad = expm(A3 * dt); xh = np.zeros(2); x = np.array([0.0, 0.0]); est_v = []
    for k in range(int(T / dt)):
        y = x[0] + noise[k]
        xh = xh + dt * (A3 @ xh + (L.ravel() * (y - xh[0])))        # observer (Euler), u = 0
        x = Ad @ x
        est_v.append(xh[1])
    print(f"observer poles at {p}: L = {L.ravel()}, rms velocity-estimate noise after 1 s = {np.std(est_v[1000:]):.4f}")

observer poles at -5: L = [ 9. 16.], rms velocity-estimate noise after 1 s = 0.0011
observer poles at -10: L = [19. 81.], rms velocity-estimate noise after 1 s = 0.0040
observer poles at -40: L = [  79. 1521.], rms velocity-estimate noise after 1 s = 0.0384


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.